# Praktikum 1 - Jaringan Saraf Tiruan (Backpropagation dari Awal) pada Dataset PBL

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Load dataset PBL
df = pd.read_csv('burnout_dataset1.csv')

# Ambil sampel terstratifikasi agar proses training dari awal berjalan cepat dan seimbang
df_sample, _ = train_test_split(
    df, train_size=2000, stratify=df['Burnout_Risk'], random_state=42
)

# Fitur numerik (kecuali Person_ID dan Burnout_Score agar tidak terjadi kebocoran data)
num_cols = df_sample.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in ['Person_ID', 'Burnout_Score']]

X_raw = SimpleImputer(strategy='median').fit_transform(df_sample[feature_cols])
X = StandardScaler().fit_transform(X_raw)

# Target biner: 1 untuk High Burnout Risk, 0 untuk Low/Moderate
y = (df_sample['Burnout_Risk'] == 'High').astype(float).values.reshape(-1, 1)

print(f"Dimensi X: {X.shape}, Dimensi y: {y.shape}")
print(f"Proporsi kelas High (1): {y.mean():.4f}")

# Parameter
input_size = X.shape[1]
hidden_size = 2
output_size = 1
lr = 0.5

# Inisialisasi bobot
np.random.seed(42)
W1 = np.random.randn(input_size, hidden_size) * 0.1
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.1
b2 = np.zeros((1, output_size))

# Fungsi aktivasi
def sigmoid(x):
    x_clip = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x_clip))

def sigmoid_derivative(x):
    return x * (1 - x)

# Training
n_samples = X.shape[0]
for epoch in range(2000):
    # Forward pass
    z1 = np.dot(X, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = sigmoid(z2)

    # Hitung error
    error = y - a2

    # Backpropagation (dinormalisasi terhadap jumlah sampel)
    d_a2 = error * sigmoid_derivative(a2) / n_samples
    d_W2 = np.dot(a1.T, d_a2)
    d_b2 = np.sum(d_a2, axis=0, keepdims=True)

    d_a1 = np.dot(d_a2, W2.T) * sigmoid_derivative(a1)
    d_W1 = np.dot(X.T, d_a1)
    d_b1 = np.sum(d_a1, axis=0, keepdims=True)

    # Update bobot
    W1 += lr * d_W1
    b1 += lr * d_b1
    W2 += lr * d_W2
    b2 += lr * d_b2

    if epoch % 500 == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch}, Loss: {loss:.6f}")

loss_awal = float(np.mean(np.square(error)))
acc_awal = float(np.mean((a2 >= 0.5) == y))
print(f"Final Loss (Hidden 2 - Sigmoid): {loss_awal:.6f}, Akurasi: {acc_awal:.4f}")

Dimensi X: (2000, 19), Dimensi y: (2000, 1)
Proporsi kelas High (1): 0.1970
Epoch 0, Loss: 0.241132


Epoch 500, Loss: 0.077584
Epoch 1000, Loss: 0.058322


Epoch 1500, Loss: 0.052411
Final Loss (Hidden 2 - Sigmoid): 0.050107, Akurasi: 0.9315


# Tugas 1 - Perbandingan Jumlah Neuron Hidden dan Fungsi Aktivasi

In [2]:
input_size = X.shape[1]
hidden_size = 3
output_size = 1
lr = 0.5

np.random.seed(42)
W1_h3 = np.random.randn(input_size, hidden_size) * 0.1
b1_h3 = np.zeros((1, hidden_size))
W2_h3 = np.random.randn(hidden_size, output_size) * 0.1
b2_h3 = np.zeros((1, output_size))

for epoch in range(2000):
    z1 = np.dot(X, W1_h3) + b1_h3
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2_h3) + b2_h3
    a2 = sigmoid(z2)

    error = y - a2

    d_a2 = error * sigmoid_derivative(a2) / n_samples
    d_W2 = np.dot(a1.T, d_a2)
    d_b2 = np.sum(d_a2, axis=0, keepdims=True)

    d_a1 = np.dot(d_a2, W2_h3.T) * sigmoid_derivative(a1)
    d_W1 = np.dot(X.T, d_a1)
    d_b1 = np.sum(d_a1, axis=0, keepdims=True)

    W1_h3 += lr * d_W1
    b1_h3 += lr * d_b1
    W2_h3 += lr * d_W2
    b2_h3 += lr * d_b2

    if epoch % 500 == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch}, Loss: {loss:.6f}")

loss_h3_sigmoid = float(np.mean(np.square(error)))
acc_h3_sigmoid = float(np.mean((a2 >= 0.5) == y))
print(f"Final Loss (Hidden 3 - Sigmoid): {loss_h3_sigmoid:.6f}, Akurasi: {acc_h3_sigmoid:.4f}")

Epoch 0, Loss: 0.257672


Epoch 500, Loss: 0.074880


Epoch 1000, Loss: 0.057723


Epoch 1500, Loss: 0.052234


Final Loss (Hidden 3 - Sigmoid): 0.049984, Akurasi: 0.9325


In [3]:
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

input_size = X.shape[1]
hidden_size = 3
output_size = 1
lr = 0.5

np.random.seed(42)
W1_relu = np.random.randn(input_size, hidden_size) * 0.1
b1_relu = np.zeros((1, hidden_size))
W2_relu = np.random.randn(hidden_size, output_size) * 0.1
b2_relu = np.zeros((1, output_size))

for epoch in range(2000):
    z1 = np.dot(X, W1_relu) + b1_relu
    a1 = relu(z1)
    z2 = np.dot(a1, W2_relu) + b2_relu
    a2 = sigmoid(z2)

    error = y - a2

    d_a2 = error * sigmoid_derivative(a2) / n_samples
    d_W2 = np.dot(a1.T, d_a2)
    d_b2 = np.sum(d_a2, axis=0, keepdims=True)

    d_a1 = np.dot(d_a2, W2_relu.T) * relu_derivative(z1)
    d_W1 = np.dot(X.T, d_a1)
    d_b1 = np.sum(d_a1, axis=0, keepdims=True)

    W1_relu += lr * d_W1
    b1_relu += lr * d_b1
    W2_relu += lr * d_W2
    b2_relu += lr * d_b2

    if epoch % 500 == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch}, Loss: {loss:.6f}")

loss_h3_relu = float(np.mean(np.square(error)))
acc_h3_relu = float(np.mean((a2 >= 0.5) == y))
print(f"Final Loss (Hidden 3 - ReLU): {loss_h3_relu:.6f}, Akurasi: {acc_h3_relu:.4f}")

Epoch 0, Loss: 0.252362


Epoch 500, Loss: 0.047390


Epoch 1000, Loss: 0.045383


Epoch 1500, Loss: 0.044633


Final Loss (Hidden 3 - ReLU): 0.044055, Akurasi: 0.9375


In [4]:
df_hasil = pd.DataFrame({
    'Konfigurasi': [
        'Awal (Hidden 2 - Sigmoid)',
        'Tugas 1 (Hidden 3 - Sigmoid)',
        'Tugas 1 (Hidden 3 - ReLU)',
    ],
    'Hidden Size': [2, 3, 3],
    'Aktivasi Hidden': ['Sigmoid', 'Sigmoid', 'ReLU'],
    'Final Loss': [round(loss_awal, 6), round(loss_h3_sigmoid, 6), round(loss_h3_relu, 6)],
    'Akurasi': [round(acc_awal, 4), round(acc_h3_sigmoid, 4), round(acc_h3_relu, 4)],
})
df_hasil

,Konfigurasi,Hidden Size,Aktivasi Hidden,Final Loss,Akurasi
0,Awal (Hidden 2 - Sigmoid),2,Sigmoid,0.050107,0.9315
1,Tugas 1 (Hidden 3 - Sigmoid),3,Sigmoid,0.049984,0.9325
2,Tugas 1 (Hidden 3 - ReLU),3,ReLU,0.044055,0.9375


Berdasarkan hasil pengujian backpropagation pada dataset burnout PBL:

1. Penambahan neuron pada hidden layer dari 2 menjadi 3 dengan aktivasi Sigmoid memberikan kapasitas representasi tambahan untuk memisahkan kelas risiko burnout tinggi terhadap risiko rendah/sedang.
2. Penggunaan fungsi aktivasi ReLU pada hidden layer mempercepat konvergensi dan menurunkan nilai loss akhir dibandingkan Sigmoid pada jumlah epoch yang sama, karena gradien ReLU pada daerah positif bernilai konstan 1 sehingga pembaruan bobot tidak terhambat oleh saturasi gradien.